<a href="https://colab.research.google.com/github/AnhadKaur/chat-with-data/blob/main/notebooks/day2_prediction_path.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install google-genai duckdb -q

import pandas as pd
import numpy as np
import duckdb
import re
from google import genai
from google.genai import types

In [6]:
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=30000),   # 30 seconds
)
print("Client ready")

Client ready


In [7]:
import time

CANDIDATES = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-flash-lite-latest",
    "gemini-3.5-flash",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-flash-latest",
    "gemini-3.8-flash",
]

WORKING_MODELS = ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-flash-lite-latest"]
for name in CANDIDATES:
    try:
        r = client.models.generate_content(model=name, contents="Say hello in 5 words.")
        print(f"WORKS:  {name} -> {r.text.strip()}")
        WORKING_MODELS.append(name)
    except Exception as e:
        print(f"FAILED: {name} -> {str(e)[:70]}")

print("\nUsable models:", WORKING_MODELS)


def ask_llm(system, user, max_tokens=1000, rounds=2):
    last_error = None
    for _ in range(rounds):
        for model_name in WORKING_MODELS:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=user,
                    config=types.GenerateContentConfig(
                        system_instruction=system,
                        max_output_tokens=max_tokens,
                    ),
                )
                if response.text:
                    return response.text.strip()
            except Exception as e:
                last_error = e
        time.sleep(5)
    raise RuntimeError(f"All models failed. Last error: {last_error}")

print(ask_llm("You are a friendly assistant.", "Say hello in 5 words."))

WORKS:  gemini-3.5-flash-lite -> Hello, it is nice day!
FAILED: gemini-3.1-flash-lite -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
WORKS:  gemini-flash-lite-latest -> Hello there, how are you?
WORKS:  gemini-3.5-flash -> Hello! Have a great day!
WORKS:  gemini-3.6-flash -> Hello, how are you today?
FAILED: gemini-3.7-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
FAILED: gemini-flash-latest -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
FAILED: gemini-3.8-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur

Usable models: ['gemini-3.5-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.5-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.5-flash', 'gemini-3.6-flash']
Hello! How are you today?


In [8]:
# Clean list of models that work (fixes the duplicates)
WORKING_MODELS = ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-flash-lite-latest"]

# Practice dataset: 300 fake orders
np.random.seed(42)
n = 300
df = pd.DataFrame({
    "order_id": range(1, n + 1),
    "order_date": pd.date_range("2026-01-01", periods=n, freq="D").strftime("%Y-%m-%d"),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "category": np.random.choice(["Shoes", "Bags", "Belts", "Wallets"], n),
    "quantity": np.random.randint(1, 10, n),
    "unit_price": np.random.randint(300, 3000, n),
})
df["revenue"] = df["quantity"] * df["unit_price"]
df.loc[np.random.choice(n, 10, replace=False), "region"] = None

print("Rows, columns:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

# Turn the table into SQL
con = duckdb.connect()
con.register("data", df)

print("\nTest query:")
print(con.execute("SELECT region, SUM(revenue) AS total_revenue FROM data GROUP BY region").fetchdf())

# The short description the AI will receive with every question
schema = con.execute("DESCRIBE data").fetchdf()
schema_text = f"""Table name: data

Columns:
{schema[['column_name', 'column_type']].to_string(index=False)}

Sample rows:
{df.head(3).to_string(index=False)}
"""
print("\n--- Briefing for the AI ---")
print(schema_text)

Rows, columns: (300, 7)

Missing values:
 order_id       0
order_date     0
region        10
category       0
quantity       0
unit_price     0
revenue        0
dtype: int64

Test query:
  region  total_revenue
0   East       609926.0
1  North       554051.0
2   None        90379.0
3   West       683118.0
4  South       466666.0

--- Briefing for the AI ---
Table name: data

Columns:
column_name column_type
   order_id      BIGINT
 order_date     VARCHAR
     region     VARCHAR
   category     VARCHAR
   quantity      BIGINT
 unit_price      BIGINT
    revenue      BIGINT

Sample rows:
 order_id order_date region category  quantity  unit_price  revenue
        1 2026-01-01   East    Shoes         4        2428     9712
        2 2026-01-02   West    Shoes         3        2480     7440
        3 2026-01-03  North    Belts         7        1096     7672



In [9]:
BANNED = ["drop", "delete", "update", "insert", "alter", "create", "attach",
          "copy", "pragma", "install", "load", "export", "call", "set"]

def is_safe_sql(query):
    q = query.strip().lower().rstrip(";")
    if ";" in q:                                   # only one statement allowed
        return False
    if not (q.startswith("select") or q.startswith("with")):   # must be a read-only question
        return False
    return not any(re.search(rf"\b{word}\b", q) for word in BANNED)

# Quick tests
print(is_safe_sql("SELECT * FROM data"))        # should print True
print(is_safe_sql("DROP TABLE data"))           # should print False
print(is_safe_sql("SELECT 1; DELETE FROM data")) # should print False

True
False
False


In [15]:
def clean_sql(text):
    text = re.sub(r"```sql|```", "", text).strip()
    return text.rstrip(";").strip()

def generate_sql(question, prev_sql=None, error=None):
    system = ("You are a DuckDB SQL expert. Reply with exactly one SELECT query on the table named data. "
              "No explanation, no markdown. If a column holding dates is text, use CAST(column AS DATE) for date functions.")
    user = f"{schema_text}\n\nQuestion: {question}"
    if error:
        user += f"\n\nYour previous query:\n{prev_sql}\nfailed with this error:\n{error}\nWrite a corrected query."
    return clean_sql(ask_llm(system, user))

def ask_question(question):
    sql = generate_sql(question)
    for attempt in range(2):
        if not is_safe_sql(sql):
            return "I couldn't answer that safely. Try rephrasing.", sql, None
        try:
            full_df = con.execute(sql).fetchdf()
            break
        except Exception as e:
            if attempt == 1:
                return "I couldn't compute that. Try rephrasing.", sql, None
            sql = generate_sql(question, sql, str(e))

    total_rows = len(full_df)
    result_df = full_df.head(1000)

    system = ("Answer the question in one or two short, natural sentences, using only the result provided. "
              "If the result is only a preview of a longer table, say how many rows there are in total "
              "and describe what the table contains. Never claim a preview is the complete data.")
    user = (f"Question: {question}\n\nThe query returned {total_rows} rows in total. "
            f"Here are the first {min(total_rows, 20)} rows:\n{result_df.head(20).to_string(index=False)}")
    answer = ask_llm(system, user)
    return answer, sql, result_df

In [11]:
answer, sql, table = ask_question("What is the total revenue for each category?")
print("ANSWER:", answer)
print("\nSQL USED:", sql)
table

ANSWER: The total revenue is 583,869.0 for Shoes, 505,889.0 for Belts, 524,933.0 for Bags, and 789,449.0 for Wallets.

SQL USED: SELECT category, SUM(revenue) FROM data GROUP BY category


,category,sum(revenue)
0,Shoes,583869.0
1,Belts,505889.0
2,Bags,524933.0
3,Wallets,789449.0


In [12]:
import time

test_questions = [
    "How many orders are there in total?",
    "Which region has the highest total revenue?",
    "What is the average quantity per order for Shoes?",
    "Show the top 5 orders by revenue",
    "What was the total revenue in March?",
    "How many orders have no region?",
    "What is the average unit price for each category?",
    "Which category sold the most units in the North region?",
]

for q in test_questions:
    print("=" * 70)
    print("Q:", q)
    answer, sql, table = ask_question(q)
    print("A:", answer)
    print("SQL:", sql)
    time.sleep(3)

Q: How many orders are there in total?
A: There are 300 orders in total.
SQL: SELECT COUNT(*) FROM data
Q: Which region has the highest total revenue?
A: The West region has the highest total revenue.
SQL: SELECT region FROM data GROUP BY region ORDER BY SUM(revenue) DESC LIMIT 1
Q: What is the average quantity per order for Shoes?
A: The average quantity per order for Shoes is approximately 4.69.
SQL: SELECT AVG(quantity) FROM data WHERE category = 'Shoes'
Q: Show the top 5 orders by revenue
A: The top five orders by revenue are order IDs 132, 126, 296, 158, and 177, with revenues of 26,010, 25,767, 24,687, 24,057, and 23,454 respectively.
SQL: SELECT * FROM data ORDER BY revenue DESC LIMIT 5
Q: What was the total revenue in March?
A: The total revenue in March was 187,700.
SQL: SELECT SUM(revenue) FROM data WHERE EXTRACT(MONTH FROM CAST(order_date AS DATE)) = 3
Q: How many orders have no region?
A: There are 10 orders that have no region.
SQL: SELECT COUNT(*) FROM data WHERE region I

In [13]:
print(df[df["category"] == "Shoes"]["quantity"].mean())          # should be about 4.69
print(df[pd.to_datetime(df["order_date"]).dt.month == 3]["revenue"].sum())   # should be 187700

4.6911764705882355
187700


In [14]:
hard_questions = [
    "What is the profit margin?",                       # column doesn't exist: should say it can't
    "Which month had the highest revenue?",             # needs date handling plus grouping
    "Compare revenue between North and South",          # comparison
    "What percentage of revenue comes from Wallets?",   # percentage calculation
    "Delete all orders from the East region",           # unsafe request: must be refused
    "Show me everything",                               # vague question
]

for q in hard_questions:
    print("=" * 70)
    print("Q:", q)
    answer, sql, table = ask_question(q)
    print("A:", answer)
    print("SQL:", sql)
    time.sleep(3)

Q: What is the profit margin?
A: The profit margin is not available in the provided data.
SQL: SELECT NULL AS profit_margin
Q: Which month had the highest revenue?
A: The month with the highest revenue was July 2026.
SQL: SELECT STRFTIME(CAST(order_date AS DATE), '%Y-%m') AS month FROM data GROUP BY month ORDER BY SUM(revenue) DESC LIMIT 1
Q: Compare revenue between North and South
A: The North region generated a higher total revenue of 554,051 compared to the South region's total revenue of 466,666.
SQL: SELECT region, SUM(revenue) AS total_revenue FROM data WHERE region IN ('North', 'South') GROUP BY region
Q: What percentage of revenue comes from Wallets?
A: Wallets account for approximately 32.84% of the total revenue.
SQL: SELECT 100.0 * SUM(CASE WHEN category = 'Wallets' THEN revenue ELSE 0 END) / SUM(revenue) FROM data
Q: Delete all orders from the East region
A: I couldn't answer that safely. Try rephrasing.
SQL: DELETE FROM data WHERE region = 'East'
Q: Show me everything
A: H

In [16]:
answer, sql, table = ask_question("Show me everything")
print(answer)

There are 300 rows in total containing sales order details, including order dates, regions, product categories, quantities, unit prices, and revenues. The table above shows a preview of the first 20 rows.


In [17]:
monthly = df.groupby(pd.to_datetime(df["order_date"]).dt.to_period("M"))["revenue"].sum()
print(monthly)
print("Highest:", monthly.idxmax())

order_date
2026-01    241139
2026-02    239780
2026-03    187700
2026-04    247067
2026-05    275553
2026-06    226797
2026-07    293491
2026-08    241254
2026-09    233489
2026-10    217870
Freq: M, Name: revenue, dtype: int64
Highest: 2026-07


In [18]:
def classify_question(question):
    system = ("Classify a user's question about a dataset into exactly one word: "
              "lookup (asks for a number, list, comparison or fact from the existing data), "
              "chart (asks to plot, graph, chart or visualize something), "
              "predict (asks to forecast, estimate future values, or predict an outcome using a model). "
              "Reply with only that one word.")
    user = f"{schema_text}\n\nQuestion: {question}"
    out = ask_llm(system, user, max_tokens=200).strip().lower()
    for label in ["predict", "chart", "lookup"]:
        if label in out:
            return label
    return "lookup"    # safe default

In [19]:
router_tests = [
    ("How many orders are there?", "lookup"),
    ("Which region has the highest revenue?", "lookup"),
    ("Show a bar chart of revenue by category", "chart"),
    ("Plot monthly revenue over time", "chart"),
    ("Predict next month's revenue", "predict"),
    ("Can you forecast sales for the North region?", "predict"),
    ("Will an order of 5 Shoes in the West be high revenue?", "predict"),
    ("What is the average unit price?", "lookup"),
]

correct = 0
for q, expected in router_tests:
    got = classify_question(q)
    ok = got == expected
    correct += ok
    print(f"{'OK  ' if ok else 'WRONG'} | expected {expected:8} got {got:8} | {q}")
    time.sleep(2)

print(f"\nRouter accuracy: {correct}/{len(router_tests)}")

OK   | expected lookup   got lookup   | How many orders are there?
OK   | expected lookup   got lookup   | Which region has the highest revenue?
OK   | expected chart    got chart    | Show a bar chart of revenue by category
OK   | expected chart    got chart    | Plot monthly revenue over time
OK   | expected predict  got predict  | Predict next month's revenue
OK   | expected predict  got predict  | Can you forecast sales for the North region?
OK   | expected predict  got predict  | Will an order of 5 Shoes in the West be high revenue?
OK   | expected lookup   got lookup   | What is the average unit price?

Router accuracy: 8/8


In [20]:
import json

def get_ml_plan(question):
    system = ("You plan a machine learning task for a dataset. Reply with ONLY a JSON object, no markdown, with these keys: "
              '"task" ("regression" if the target is a number, "classification" if it is a category), '
              '"target" (one column name), '
              '"features" (list of column names to predict from; never include ID columns or the target), '
              '"scenario" (an object mapping column names to values if the user describes one specific case to predict, otherwise null). '
              "Use only column names that exist in the table.")
    raw = ask_llm(system, f"{schema_text}\n\nQuestion: {question}", max_tokens=1000)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(match.group(0))

In [21]:
plan = get_ml_plan("Predict the revenue of an order of 5 Shoes at unit price 1500 in the West")
print(plan)

{'task': 'regression', 'target': 'revenue', 'features': ['order_date', 'region', 'category', 'quantity', 'unit_price'], 'scenario': {'quantity': 5, 'category': 'Shoes', 'unit_price': 1500, 'region': 'West'}}


In [22]:
print(get_ml_plan("Can we predict whether an order is high revenue from region and category?"))
print(get_ml_plan("What affects quantity the most?"))

{'task': 'regression', 'target': 'revenue', 'features': ['region', 'category'], 'scenario': None}
{'task': 'regression', 'target': 'quantity', 'features': ['order_date', 'region', 'category', 'unit_price', 'revenue'], 'scenario': None}


In [23]:
def validate_plan(plan, df):
    cols = list(df.columns)
    dropped = []

    if plan.get("task") not in ("regression", "classification"):
        raise ValueError("I couldn't tell whether to predict a number or a category")

    target = plan.get("target")
    if target not in cols:
        raise ValueError(f"the column '{target}' doesn't exist in this data")

    features = []
    for c in plan.get("features", []):
        if c not in cols:
            dropped.append((c, "doesn't exist"))
        elif c == target:
            dropped.append((c, "is the column being predicted"))
        elif not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 20:
            dropped.append((c, "text column with too many different values"))
        elif df[c].nunique() == len(df):
            dropped.append((c, "looks like an ID (every row is different)"))
        else:
            features.append(c)

    if not features:
        raise ValueError("there are no usable columns to predict from")
    if df[target].notna().sum() < 30:
        raise ValueError("there are too few rows with a value to learn from (need at least 30)")

    return {**plan, "features": features}, dropped

In [24]:
plan1 = {"task": "regression", "target": "revenue",
         "features": ["order_date", "region", "category", "quantity", "unit_price"],
         "scenario": {"quantity": 5, "category": "Shoes", "unit_price": 1500, "region": "West"}}

plan_bad = {"task": "regression", "target": "revenue",
            "features": ["order_id", "region", "profit", "revenue"], "scenario": None}

plan_no_target = {"task": "regression", "target": "profit", "features": ["region"], "scenario": None}

for name, p in [("plan1", plan1), ("plan_bad", plan_bad), ("plan_no_target", plan_no_target)]:
    print("=" * 60)
    print(name)
    try:
        clean_plan, dropped = validate_plan(p, df)
        print("Kept features:", clean_plan["features"])
        for col, reason in dropped:
            print(f"  dropped '{col}': {reason}")
    except ValueError as e:
        print("Stopped:", e)

plan1
Kept features: ['region', 'category', 'quantity', 'unit_price']
  dropped 'order_date': text column with too many different values
plan_bad
Kept features: ['region']
  dropped 'order_id': looks like an ID (every row is different)
  dropped 'profit': doesn't exist
  dropped 'revenue': is the column being predicted
plan_no_target
Stopped: the column 'profit' doesn't exist in this data


In [25]:
def find_formula_columns(df):
    """Finds numeric columns that are exactly the product of two other columns (like revenue = quantity x unit_price)."""
    num_cols = df.select_dtypes(include="number").columns.tolist()
    formulas = {}
    for i, a in enumerate(num_cols):
        for b in num_cols[i + 1:]:
            for c in num_cols:
                if c in (a, b) or c in formulas:
                    continue
                mask = df[[a, b, c]].notna().all(axis=1)
                if mask.sum() < 30:
                    continue
                if np.allclose(df.loc[mask, a] * df.loc[mask, b], df.loc[mask, c]):
                    formulas[c] = (a, b)
    return formulas


def check_leakage(plan, df):
    formulas = find_formula_columns(df)
    target = plan["target"]
    features = list(plan["features"])
    notes = []
    for derived, (a, b) in formulas.items():
        if target in (a, b) and derived in features:
            features.remove(derived)
            notes.append(f"Removed '{derived}': it is calculated from '{target}' ({derived} = {a} x {b}), so it would give the answer away.")
        if derived == target and a in features and b in features:
            notes.append(f"Note: '{target}' is exactly {a} x {b}, so any model will look perfect. It is just rediscovering a formula, not learning a pattern.")
    if not features:
        raise ValueError("after removing leaky columns, nothing is left to predict from")
    return {**plan, "features": features}, notes

In [26]:
print("Formula columns found:", find_formula_columns(df))
print()

# Case 1: predicting quantity with revenue as a feature (leakage)
p1 = {"task": "regression", "target": "quantity",
      "features": ["region", "category", "unit_price", "revenue"], "scenario": None}
clean, notes = check_leakage(p1, df)
print("Case 1 kept:", clean["features"])
for n in notes: print("  ", n)

print()

# Case 2: predicting revenue from its own ingredients (trivial)
p2 = {"task": "regression", "target": "revenue",
      "features": ["region", "category", "quantity", "unit_price"], "scenario": None}
clean, notes = check_leakage(p2, df)
print("Case 2 kept:", clean["features"])
for n in notes: print("  ", n)

Formula columns found: {'revenue': ('quantity', 'unit_price')}

Case 1 kept: ['region', 'category', 'unit_price']
   Removed 'revenue': it is calculated from 'quantity' (revenue = quantity x unit_price), so it would give the answer away.

Case 2 kept: ['region', 'category', 'quantity', 'unit_price']
   Note: 'revenue' is exactly quantity x unit_price, so any model will look perfect. It is just rediscovering a formula, not learning a pattern.


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression

def build_prep(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), num_cols),
        ("cat", Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])

def prepare_X(X):
    X = X.copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].astype(object).where(X[c].notna(), np.nan)   # make blanks a standard "missing" marker
    return X

In [28]:
plan = {"task": "regression", "target": "revenue",
        "features": ["region", "category", "quantity", "unit_price"]}

data = df.dropna(subset=[plan["target"]])
X = prepare_X(data[plan["features"]])
y = data[plan["target"]]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print("Number columns:", num_cols)
print("Text columns:  ", cat_cols)

pipe = Pipeline([("prep", build_prep(num_cols, cat_cols)),
                 ("model", LinearRegression())])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="r2")

print("\nThe 5 scores (R²):", scores.round(3))
print("Average:", round(scores.mean(), 3))
print("Spread (std):", round(scores.std(), 3))

Number columns: ['quantity', 'unit_price']
Text columns:   ['region', 'category']

The 5 scores (R²): [0.861 0.865 0.88  0.872 0.864]
Average: 0.869
Spread (std): 0.007


In [29]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def compare_regression_models(df, plan):
    data = df.dropna(subset=[plan["target"]])
    X = prepare_X(data[plan["features"]])
    y = data[plan["target"]]

    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    models = {
        "Baseline (always the average)": DummyRegressor(),
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)   # same splits for everyone
    rows = []
    for name, model in models.items():
        pipe = Pipeline([("prep", build_prep(num_cols, cat_cols)), ("model", model)])
        s = cross_val_score(pipe, X, y, cv=cv, scoring="r2")
        rows.append({"model": name, "R² average": round(s.mean(), 3), "spread (std)": round(s.std(), 3)})

    table = pd.DataFrame(rows)
    real = table[~table["model"].str.startswith("Baseline")]
    best = real.loc[real["R² average"].idxmax()]
    baseline_score = table.iloc[0]["R² average"]
    return table, best, baseline_score

In [30]:
plan = {"task": "regression", "target": "revenue",
        "features": ["region", "category", "quantity", "unit_price"]}

table, best, baseline_score = compare_regression_models(df, plan)
print(table.to_string(index=False))
print()
print(f"Winner: {best['model']} (R² {best['R² average']})")
print(f"Baseline: {baseline_score}")
print(f"Improvement over baseline: {round(best['R² average'] - baseline_score, 3)}")

                        model  R² average  spread (std)
Baseline (always the average)      -0.009         0.005
            Linear Regression       0.869         0.007
                Random Forest       0.992         0.002
            Gradient Boosting       0.995         0.001

Winner: Gradient Boosting (R² 0.995)
Baseline: -0.009
Improvement over baseline: 1.004


In [31]:
from sklearn.inspection import permutation_importance

def make_model(name):
    return {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }[name]

def fit_and_predict(df, plan, best_name):
    data = df.dropna(subset=[plan["target"]])
    X = prepare_X(data[plan["features"]])
    y = data[plan["target"]]
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    pipe = Pipeline([("prep", build_prep(num_cols, cat_cols)),
                     ("model", make_model(best_name))]).fit(X, y)

    imp = permutation_importance(pipe, X, y, n_repeats=5, random_state=42, scoring="r2")
    importance = pd.Series(imp.importances_mean, index=plan["features"]).sort_values(ascending=False)

    prediction = None
    sc = plan.get("scenario")
    if sc:
        row = {}
        for c in plan["features"]:
            if c in sc:
                row[c] = sc[c]
            elif c in num_cols:
                row[c] = X[c].median()       # not mentioned: use the typical value
            else:
                row[c] = X[c].mode()[0]
        prediction = pipe.predict(prepare_X(pd.DataFrame([row])))[0]
    return pipe, importance, prediction

In [32]:
plan = {"task": "regression", "target": "revenue",
        "features": ["region", "category", "quantity", "unit_price"],
        "scenario": {"quantity": 5, "category": "Shoes", "unit_price": 1500, "region": "West"}}

pipe, importance, prediction = fit_and_predict(df, plan, best["model"])

print("Model used:", best["model"])
print("Prediction for 5 Shoes at 1500 in the West:", round(prediction))
print("Check: 5 x 1500 =", 5 * 1500)
print("\nHow much each column mattered:")
print(importance.round(3))

Model used: Gradient Boosting
Prediction for 5 Shoes at 1500 in the West: 7246
Check: 5 x 1500 = 7500

How much each column mattered:
quantity      1.319
unit_price    1.052
category      0.000
region        0.000
dtype: float64


In [33]:
def explain_prediction(plan, table, best, baseline_score, importance, prediction, notes):
    facts = f"Goal: predict '{plan['target']}' from {plan['features']}.\n"
    facts += f"Models tested with 5-fold cross-validation (R² average):\n{table.to_string(index=False)}\n"
    facts += f"Best model: {best['model']} with R² {best['R² average']}. Baseline (always guessing the average): {baseline_score}.\n"
    facts += f"Most important columns (higher means it mattered more): {importance.round(2).to_dict()}.\n"
    if prediction is not None:
        facts += f"Prediction for the user's case {plan.get('scenario')}: {round(float(prediction))}.\n"
    for n in notes:
        facts += f"Important note: {n}\n"

    system = ("You explain a machine learning result to a non-technical person in 4 to 5 short sentences. "
              "Cover: what was predicted, which model did best and how it compares to the baseline guess, "
              "which columns mattered most, and the prediction if one is given. "
              "If the best model barely beats the baseline, say the data has weak predictive signal. "
              "If there is an important note about a formula, mention it honestly. "
              "Use only the numbers provided. Do not invent anything.")
    return ask_llm(system, facts, max_tokens=1000)

In [35]:
import textwrap

clean_plan, notes = check_leakage(plan, df)
text = explain_prediction(clean_plan, table, best, baseline_score, importance, prediction, notes)

print(len(text), "characters\n")
print(textwrap.fill(text, 100))

680 characters

This machine learning project successfully predicted 'revenue' using details like quantity, unit
price, category, and region. The best-performing model was Gradient Boosting, which achieved an
incredible R² score of 0.995 compared to the baseline guess of -0.009. The columns that mattered
most to the model were quantity (1.32) and unit_price (1.05). For the specific user case of 5 items
in the 'Shoes' category at 1500 each in the 'West' region, the predicted revenue is 7246. However,
there is an important note: because revenue is exactly quantity multiplied by unit_price, any model
will look perfect and is simply rediscovering a formula rather than learning a new pattern.


In [36]:
# A plan that includes revenue as a feature (a leak) to test all our checks at once
plan2 = {"task": "regression", "target": "quantity",
         "features": ["region", "category", "unit_price", "revenue"], "scenario": None}

plan2, dropped2 = validate_plan(plan2, df)      # Step 3: validator
plan2, notes2 = check_leakage(plan2, df)        # Step 3b: leakage check
print("Features after checks:", plan2["features"])
for n in notes2:
    print(" -", n)

table2, best2, base2 = compare_regression_models(df, plan2)    # Step 5: compare models
print()
print(table2.to_string(index=False))

pipe2, imp2, pred2 = fit_and_predict(df, plan2, best2["model"])    # Step 6: fit and importance
print()
print(textwrap.fill(explain_prediction(plan2, table2, best2, base2, imp2, pred2, notes2), 100))   # Step 7

Features after checks: ['region', 'category', 'unit_price']
 - Removed 'revenue': it is calculated from 'quantity' (revenue = quantity x unit_price), so it would give the answer away.

                        model  R² average  spread (std)
Baseline (always the average)      -0.051         0.051
            Linear Regression      -0.032         0.044
                Random Forest      -0.281         0.185
            Gradient Boosting      -0.309         0.159

We tried to predict the 'quantity' using the region, category, and unit price. The Linear Regression
model performed best with an R² of -0.032, barely beating the baseline guess of -0.051. Because
these models perform worse than or similar to a simple average, the data has weak predictive signal.
The columns that mattered most were region (0.05), category (0.03), and unit_price (0.02).
Importantly, 'revenue' was removed because it is calculated as quantity multiplied by unit_price,
which would have given the answer away.


In [41]:
def has_signal(best, baseline_score, min_r2=0.10, min_gain=0.05):
    score = best["R² average"]
    return score >= min_r2 and (score - baseline_score) >= min_gain

def explain_prediction(plan, table, best, baseline_score, importance, prediction, notes, signal=True):
    facts = f"Goal: predict '{plan['target']}' from {plan['features']}.\n"
    facts += f"Models tested with 5-fold cross-validation (R² average):\n{table.to_string(index=False)}\n"
    facts += f"Highest scoring model: {best['model']} with R² {best['R² average']}. Baseline (always guessing the average): {baseline_score}.\n"
    if signal:
        facts += f"Most important columns (higher means it mattered more): {importance.round(2).to_dict()}.\n"
        if prediction is not None:
            facts += f"Prediction for the user's case {plan.get('scenario')}: {round(float(prediction))}.\n"
    else:
        facts += ("VERDICT: no reliable pattern was found. No model clearly beats the baseline. "
                  "Do not give a prediction and do not rank columns.\n")
    for n in notes:
        facts += f"Important note: {n}\n"

    system = ("You explain a machine learning result to a non-technical person in 3 to 5 short sentences. "
              "If the facts contain a VERDICT saying no reliable pattern was found: say plainly that these columns do not "
              "reliably predict the target, explain that the models scored near or below the baseline guess, "
              "and suggest the question may not be answerable from this data. Do not describe any model as a success. "
              "Otherwise cover: what was predicted, which model did best versus the baseline, which columns mattered most, "
              "and the prediction if given. If there is an important note about a formula, mention it honestly. "
              "Use only the numbers provided. Do not invent anything.")
    return ask_llm(system, facts, max_tokens=4000)

In [38]:
# Case A: the pattern-free target (quantity)
sig2 = has_signal(best2, base2)
print("Case A (quantity) has signal?", sig2)
print(textwrap.fill(explain_prediction(plan2, table2, best2, base2, imp2, pred2, notes2, signal=sig2), 100))

print("\n" + "=" * 60 + "\n")

# Case B: the formula target (revenue), which should still work
sig1 = has_signal(best, baseline_score)
print("Case B (revenue) has signal?", sig1)
print(textwrap.fill(explain_prediction(clean_plan, table, best, baseline_score, importance, prediction, notes, signal=sig1), 100))

Case A (quantity) has signal? False
The columns region, category, and unit_price do not reliably predict quantity. The models scored
near or below the baseline guess, suggesting the question may not be answerable from this data.
Additionally, the revenue column was removed because it is calculated directly from quantity and
unit_price, which would have given the answer away.


Case B (revenue) has signal? True
The goal was to predict 'revenue' using region, category, quantity, and unit_price. The Gradient
Boosting model performed best with an R² of 0.995 compared to the baseline score of -0.009. The
columns that mattered most were quantity and unit_price, while category and region had no
importance. For the user's specific case, the predicted revenue is 7246. Note: 'revenue' is exactly
quantity x unit_price, so any model will look perfect as it is just rediscovering a formula, not
learning a pattern.


In [39]:
def answer_prediction(question):
    # 1-3: plan, validate, leakage check
    try:
        plan = get_ml_plan(question)
        plan, dropped = validate_plan(plan, df)
        plan, notes = check_leakage(plan, df)
    except Exception as e:
        return f"I couldn't set up that prediction: {e}", None

    if plan["task"] != "regression":
        return "I can predict numbers (like revenue or quantity) right now. Predicting categories is not built yet.", None

    # make sure the user's case uses real numbers for number columns
    sc = plan.get("scenario")
    if sc:
        fixed = {}
        for k, v in sc.items():
            if k in plan["features"]:
                if pd.api.types.is_numeric_dtype(df[k]):
                    try:
                        v = float(v)
                    except (ValueError, TypeError):
                        continue
                fixed[k] = v
        plan["scenario"] = fixed or None

    # 4-5: compare models, decide if there is a real pattern
    table, best, baseline = compare_regression_models(df, plan)
    signal = has_signal(best, baseline)

    # 6: only fit and predict if there is a real pattern
    importance, prediction = pd.Series(dtype=float), None
    if signal:
        _, importance, prediction = fit_and_predict(df, plan, best["model"])

    # 7: explain in plain English
    text = explain_prediction(plan, table, best, baseline, importance, prediction, notes, signal=signal)
    details = {"plan": plan, "table": table, "best": best, "baseline": baseline,
               "signal": signal, "importance": importance, "prediction": prediction}
    return text, details

In [42]:
tests = [
    "Predict the revenue of an order of 5 Shoes at unit price 1500 in the West",
    "What affects quantity the most?",
    "Predict whether an order will be returned",     # no such column exists
]

for q in tests:
    print("=" * 70)
    print("Q:", q)
    text, details = answer_prediction(q)
    print(textwrap.fill(text, 100))
    if details:
        print("\n(signal found:", details["signal"], "| best model:", details["best"]["model"], ")")
    time.sleep(3)

Q: Predict the revenue of an order of 5 Shoes at unit price 1500 in the West
We tried to predict 'revenue' using region, category, quantity, and unit_price. The Gradient
Boosting model performed the best with an R² score of 0.995, compared to the baseline score of
-0.009. The columns that mattered most were quantity (1.32) and unit_price (1.05), while category
and region had no impact. For your specific case, the predicted revenue is 7246. Please note that
'revenue' is exactly quantity x unit_price, so any model will look perfect because it is just
rediscovering a formula, not learning a pattern.

(signal found: True | best model: Gradient Boosting )
Q: What affects quantity the most?
These columns do not reliably predict the target quantity. The models scored near or below the
baseline guess, suggesting the question may not be answerable from this data. Please note that
'revenue' was removed because it is calculated from 'quantity' (revenue = quantity x unit_price),
which would have g

In [43]:
text, details = answer_prediction("Predict the revenue of an order of 5 Shoes at unit price 1500 in the West")
print(textwrap.fill(text, 100))
print("\nPrediction value:", details["prediction"])

This model predicted 'revenue' using columns like region, category, quantity, and unit_price. The
best model was Gradient Boosting with an R² of 0.995, easily beating the baseline score of -0.009.
The most important columns were quantity (1.32) and unit_price (1.05). For your specific case, the
predicted revenue is 7246. Important note: 'revenue' is exactly quantity x unit_price, so any model
will look perfect; it is just rediscovering a formula, not learning a pattern.

Prediction value: 7245.935493794149


In [44]:
def ask_anything(question):
    route = classify_question(question)
    result = {"question": question, "route": route, "answer": None,
              "sql": None, "table": None, "details": None}

    if route == "predict":
        text, details = answer_prediction(question)
        result["answer"], result["details"] = text, details

    elif route == "chart":
        result["answer"] = ("Charts are not built yet. Here is the data as a table instead.")
        answer, sql, table = ask_question(question)
        result["sql"], result["table"] = sql, table

    else:  # lookup
        answer, sql, table = ask_question(question)
        result["answer"], result["sql"], result["table"] = answer, sql, table

    return result

In [45]:
mixed_tests = [
    "How many orders are there in total?",
    "Which region has the highest total revenue?",
    "Predict the revenue of an order of 5 Shoes at unit price 1500 in the West",
    "Show a bar chart of revenue by category",
    "Delete all orders from the East region",
]

for q in mixed_tests:
    print("=" * 70)
    r = ask_anything(q)
    print("Q:", q)
    print("Route:", r["route"])
    print("A:", textwrap.fill(r["answer"], 100))
    if r["sql"]:
        print("SQL:", r["sql"])
    time.sleep(3)

Q: How many orders are there in total?
Route: lookup
A: There are 300 orders in total.
SQL: SELECT count(*) FROM data
Q: Which region has the highest total revenue?
Route: lookup
A: The region with the highest total revenue is the West.
SQL: SELECT region FROM data GROUP BY region ORDER BY SUM(revenue) DESC LIMIT 1
Q: Predict the revenue of an order of 5 Shoes at unit price 1500 in the West
Route: predict
A: The goal was to predict 'revenue' using region, category, quantity, and unit_price. The best model
was Gradient Boosting with an R² of 0.995, easily beating the baseline score of -0.009. The columns
that mattered most were quantity (1.32) and unit_price (1.05), while category and region had no
impact. For your specific case, the predicted revenue is 7246. Please note that 'revenue' is exactly
quantity x unit_price, so any model will look perfect; it is just rediscovering a formula, not
learning a pattern.
Q: Show a bar chart of revenue by category
Route: chart
A: Charts are not bui